Funded by the EU

[EU Project Website](https://ec.europa.eu/info/funding-tenders/opportunities/portal/screen/opportunities/projects-details/43252476/101261486)


# Use openEO CDSE access point to retrieve S2-based NDMI and REP time series

**Replication of the HSL pipeline**

This notebook covers the acquisition and preprocessing of satellite data for the project COSEO. The preprocessing pipeline has been developed by HSL. In this notebook, the download of satellite imagery and subsequent processing of NDMI and REP, as well as the weekly spatial aggregation to parcels is moved to cloud processor openEO. What is missing is the imputation of weekly values through machine learning. 

A CDSE account offers access to two different providers of openEO: Original (https://openeo.dataspace.copernicus.eu/) and Federation (https://openeofed.dataspace.copernicus.eu/). All available providers and the dedicated datasets, services and functions they offer can be investigated here at the OpenEO HUb https://hub.openeo.org/. Since Sentinel-2 data is available in each of them, the Original access should suffice. For one CDSE account, there are 10000 credits available per month.

To access the openEO package, we use the ``openEO`` kernel of the CDSE JupyterHub (https://jupyterhub.dataspace.copernicus.eu/). Since Geopandas is not readily available, we install it additionally.

In [ ]:
#!pip install geopandas

In [ ]:
import openeo
import os
import json

import pandas as pd
import geopandas as gpd
from shapely.geometry import box

## Load field boundaries

In [ ]:
# all fields you want to process
in_fields = "path/to/your/fields.gpkg"
gdf = gpd.read_file(in_fields)

# In this folder all jobs result will be saved in separate folders named by job id (i.e. job_j-2610160451064148974d6dddfea73543) and the job tracker
RESULT_DIR = ("path/to/result/folder")

In [ ]:
def ask_unique_id(gdf: gpd.GeoDataFrame) -> str:
    """
    Finds candidate columns for a unique ID.
    This is necessary to rejoin the result from openEO to the original data
    """
    candidates = [
        c for c in gdf.columns
        if c != gdf.geometry.name and gdf[c].notna().all() and gdf[c].is_unique
    ]
    if not candidates:
        raise ValueError("No column in this GeoDataFrame contains unique values.")

    print("Columns that qualify as a unique ID:")
    for i, c in enumerate(candidates):
        print(f"  [{i}] {c}")

    while True:
        choice = input("Select number (or column name): ").strip()
        if choice.isdigit() and int(choice) < len(candidates):
            return candidates[int(choice)]
        if choice in candidates:
            return choice
        print("Invalid choice, try again.")

In [ ]:
id_col = ask_unique_id(gdf)

In [ ]:
# values from sentinel-2 imagery may be problematic if a field is very small. therefore, we employ a 10 m negative buffer and only keep the fields of a certain size

BUFFER_M = -10
PIXEL_AREA_10M = 10 * 10      # m²
MIN_PIXELS = 2                # tune: how many "core" pixels are the minimum

gdf = gdf.to_crs(3035)
gdf["geometry"] = gdf.geometry.make_valid()

gdf["geom_orig"] = gdf.geometry
buf = gdf.geometry.buffer(BUFFER_M)          # fields thinner than 20 m become empty
gdf["geom_buf"] = buf

gdf["area_orig_m2"] = gdf["geom_orig"].area
gdf["area_buf_m2"] = buf.area
gdf["area_ratio"] = gdf["area_buf_m2"] / gdf["area_orig_m2"]

# The index: 1 = use in processing, 0 = skip
gdf["use_s2"] = (
    ~buf.is_empty
    & (gdf["area_buf_m2"] >= MIN_PIXELS * PIXEL_AREA_10M)
).astype("uint8")

# Use the buffered geometry for processing, keep the originals for reference
proc = gdf[gdf["use_s2"] == 1].set_geometry("geom_buf").drop(columns="geom_orig")

# Note: the currently unprocessed fields are not saved explicitly. They will be visible once the final result is matched with the original df and no matches show.

## Create MultiJob Database for tile wise processing

The amount of fields might not be a problem, but the geographical area spans more than recommended by openEO (~100km² for one job maximum). Therefore, we create a tiling-grid and job database to process tile by tile.

In [ ]:
from openeo.extra.job_management import (
    MultiBackendJobManager,
    create_job_db,
    split_area,
)

In [ ]:
# rough area of interest NL and Germany - if running outside this area, adapt the AOI accordingly
AREA_OF_INTEREST = {
    "west": 3_690_000.0,
    "south": 2_660_000.0,
    "east": 4_772_000.0,
    "north": 3_590_000.0,
    "crs": "EPSG:3035",
}
TILE_SIZE = 10_000  # 10 km in meters

In [ ]:
# now we split the area of interest into tiles
jobs_database = split_area(
    aoi=AREA_OF_INTEREST,
    tile_size=TILE_SIZE,
    projection="EPSG:3035",
)

# We add a tile_id column to the database for easier reference to the tiles when creating and managing jobs
jobs_database["tile_id"] = jobs_database.index

In [ ]:
# mask jobs_database based on available fields
jobs_database = jobs_database[jobs_database.intersects(proc.union_all())]

In [ ]:
JOB_TRACKER = os.path.join(RESULT_DIR, "job_tracker.csv")

# columns in the job tracker
JOB_ID_COLUMN = "id"
TILE_ID_COLUMN = "tile_id"
PROC_STATUS = "status"

In [ ]:
sidx = proc.sindex                      # only those fields that are large enough
tile_field_mapping = {}
assigned = set()

for _, row in jobs_database.iterrows():
    bbox = box(*row.geometry.bounds)
    members = proc.iloc[sidx.query(bbox, predicate="covers")]   # bbox fully covers the field
    if members.empty:
        continue                                               # no job for empty tiles

    assigned.update(members[id_col])
    tile_field_mapping[row["tile_id"]] = (                    # keep mapping of original id_col in relation to the tile id, so we can rejoin later
        members[[id_col]]
        .reset_index(drop=True)
        .rename_axis("feature_id")
        .reset_index()
    )

leftover_fields = proc[~proc[id_col].isin(assigned)]
jobs_pass0 = jobs_database[jobs_database["tile_id"].isin(tile_field_mapping)]

print(len(proc) - len(leftover_fields), "fields assigned,", len(leftover_fields), "left over")

In [ ]:
import pickle
# keep tile field mapping for rejoin with original dataframe
with open(os.path.join(RESULT_DIR, "tile_field_mapping_pass0.pkl"), "wb") as f:
    pickle.dump(tile_field_mapping, f)

## Prepare jobs and create process graphs of openEO

In [ ]:
def start_job(row: pd.Series, connection: openeo.Connection, **_) -> openeo.BatchJob:
    """
    Start a new job for a single split_area tile. Record the fields that connect to this tile
    for matching later on. This also includes the complete 
    """
    bbox = box(*row.geometry.bounds)
    west, south, east, north = row.geometry.bounds

    mapping = tile_field_mapping[row["tile_id"]]
    
    fields_in_tile = (
        proc[proc[id_col].isin(mapping[id_col])]
        .set_index(id_col)
        .loc[mapping[id_col]]
        .reset_index()
    )

    fields_wgs = fields_in_tile.to_crs(4326)
    geometries = json.loads(fields_wgs.geometry.to_json())
    
    spatial_extent = {
        "west": west,
        "south": south,
        "east": east,
        "north": north,
        "crs": "EPSG:3035",
    }

    s2cube = connection.load_collection(
        collection_id="SENTINEL2_L2A",
        spatial_extent=spatial_extent,
        temporal_extent=["2026-04-26", "2026-08-02"],
        bands=["B04", "B05", "B06", "B07", "B08", "B11", "SCL"],
        properties={"eo:cloud_cover": lambda x: x.lte(70)},
    )

    scl_band = s2cube.band("SCL")
    flat_mask = (scl_band != 4) & (scl_band != 5)
    
    red = s2cube.band("B04")     # ~665 nm
    re1 = s2cube.band("B05")     # ~705 nm
    re2 = s2cube.band("B06")     # ~740 nm
    re3 = s2cube.band("B07")     # ~783 nm
    nir = s2cube.band("B08")
    swir1 = s2cube.band("B11")

    ndmi = (nir - swir1) / (nir + swir1)
    rep = 700 + 40 * (((red + re3) / 2.0 - re1) / (re2 - re1))

    ndmi_masked = ndmi.mask(flat_mask)
    rep_masked = rep.mask(flat_mask)

    indices = ndmi_masked.add_dimension(
        name="bands", label="NDMI", type="bands"
    ).merge_cubes(
        rep_masked.add_dimension(name="bands", label="REP", type="bands")
    )
    
    weekly_cube = indices.aggregate_temporal_period(period="week", reducer="mean")
    
    timeseries = weekly_cube.aggregate_spatial(
        geometries=geometries, reducer="mean"
    )
    
    return timeseries.create_job(
        title=f"MultiBackendJobManager - Tile {row[TILE_ID_COLUMN]}",
        out_format="CSV",
    )

In [ ]:
# create the job database based on tiles where fields fall inside
job_db = create_job_db(JOB_TRACKER, df=jobs_pass0, on_exists="skip")

# Initiate MultiBackendJobManager
manager = MultiBackendJobManager(
    root_dir=RESULT_DIR, 
    download_results=True,)

connection = openeo.connect(url="openeo.dataspace.copernicus.eu").authenticate_oidc() # click on the link and enter your details
manager.add_backend("cdse", connection=connection, parallel_jobs=2)

# Run the jobs
manager.run_jobs(job_db=job_db, start_job=start_job)

print("All jobs have finished running.") # may run really long, currently no tracker

In [ ]:
from openeo.extra.job_management import CsvJobDatabase

db0 = CsvJobDatabase(JOB_TRACKER)
df0 = db0.read() # will show errors if there are any

In [ ]:
sum(df0["costs"])

In [ ]:
print(df0.loc[df0.status.isin(["error", "start_failed", "skipped"]),
             ["id", "status", "backend_name"]])

In [ ]:
# Reset the failed tile(s), does nothing if all completed successfully
mask = df0.status.isin(["error", "start_failed"]) 
df0.loc[mask, "status"] = "not_started"
df0.loc[mask, "id"] = None                             # drop the old openEO job ID
df0.loc[mask, "start_time"] = None                     

db0.persist(df0)

In [ ]:
# Run the jobs again, does nothing if all completed successfully
manager.run_jobs(job_db=job_db, start_job=start_job)

print("All jobs have finished running.")

# Managing the leftover fields

During the processing, there are fields that are split between two tiles. Currently, we do not have a solution for these fields yet.

##  Shifting the tile based database

Currently this is a replication of the process before and needs to be integrated once tests have been concluded

**currently not ready**

If there only a small number of fields missing, but they are allocated over the whole area, they would create another round of lots of jobs, even if only a small area is covered. This is a huge misallocation of resources. Other approaches shall be tested for these fields. 

# Rejoin fields

As mentioned earlier, in the processing the unique ID of the fields was lost, which is why we created the "tile_field_mapping". We can now rejoin them.

In [ ]:
with open("tile_field_mapping_pass0.pkl", "rb") as f:
    tile_field_mapping = pickle.load(f)

In [ ]:
from pathlib import Path

db0 = CsvJobDatabase(JOB_TRACKER)
finished = db0.read().query("status == 'finished'")

all_results = []

for _, job_row in finished.iterrows():
    tile_id = job_row[TILE_ID_COLUMN]
    job_id = job_row["id"]

    csv_path = Path(RESULT_DIR) / f"job_{job_id}" / "timeseries.csv"
    if not csv_path.exists():
        print(f"No CSV for Tile {tile_id} (Job {job_id}) found, skipped.")
        continue

    result = pd.read_csv(csv_path)
    mapping = tile_field_mapping[tile_id]

    merged = result.merge(
        mapping,
        left_on="feature_index",
        right_on="feature_id",
        how="left",
        validate="many_to_one",   # mapping must be unique per feature_id
    )

    unmatched = merged[id_col].isna().sum()
    if unmatched > 0:
        print(f"{unmatched} rows without {id_col}-assignment in Tile {tile_id}")

    missing = set(mapping[id_col]) - set(merged[id_col].dropna())
    if missing:
        print(f"{len(missing)} fields without result in Tile {tile_id}")

    merged["tile_id"] = tile_id
    all_results.append(merged)

final_df = pd.concat(all_results, ignore_index=True)
final_df = final_df.drop(columns=["feature_id", "feature_index"])

final_df.to_csv(Path(RESULT_DIR) / f"timeseries_test.csv", index=False)

This concludes the EO preprocessing. All next steps are done on tables with additional scripts. 